# 03 — Machine Learning Model (LightGBM)

SARIMA set a real bar in `02_baselines.ipynb` (~4.5% MAPE). SARIMA can only
see the load series itself, though — it can't directly use temperature or
holiday flags. LightGBM can use all of that plus the lag/rolling/calendar
features from `src/features.py`. The question this notebook answers: does
that extra information actually translate into a real improvement, or is
SARIMA's built-in seasonal structure hard to beat regardless?


In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
import shap

from src.features import build_feature_set
from src.evaluation import run_backtest, summarize

plt.rcParams["figure.figsize"] = (14, 4)

df = pd.read_csv("../data/processed/load_weather_hourly.csv", index_col=0, parse_dates=True)
df.index.name = "timestamp"
df = df.sort_index()
df.head()

## 1. Feature set

`build_feature_set` (from `src/features.py`) adds lag features (24h, 48h,
168h — chosen from the ACF/PACF findings in `01_eda.ipynb`), rolling
mean/std, and cyclical calendar encodings, then drops the warm-up rows that
don't have full lag history yet.


In [ ]:
feat_preview = build_feature_set(df)
print(feat_preview.shape)
feat_preview.columns.tolist()

## 2. LightGBM inside the walk-forward harness

This reuses `run_backtest` from `src/evaluation.py` — the exact same harness
SARIMA and the naive baselines ran through, so the comparison is apples to
apples.

**Important detail:** features are built on `train_df` and `test_df`
*combined*, not separately. This isn't a leakage risk — every feature here
(lags, rolling stats) only looks *backward* in time via `.shift()`, so a
test-set row's features only ever use information that was already known
before that timestamp (either from the training period, or from earlier
rows within the test window itself). Building them together just avoids
recomputing lag history redundantly and ensures the lag windows have enough
prior data available.


In [ ]:
FEATURE_DROP_COLS = ["load_MW", "holiday_name"]  # holiday_name is a string label, not a usable numeric feature as-is

def lightgbm_forecast(train_df, test_df):
    combined = pd.concat([train_df, test_df])
    feat = build_feature_set(combined)
    feature_cols = [c for c in feat.columns if c not in FEATURE_DROP_COLS]

    train_feat = feat.loc[feat.index.isin(train_df.index)]
    test_feat = feat.loc[feat.index.isin(test_df.index)]
    # Guard: reindex to the full test window and fill any edge gaps, so a
    # dropped row never silently shrinks the prediction array out of sync
    # with y_true in the harness.
    test_feat = test_feat.reindex(test_df.index).ffill().bfill()

    model = lgb.LGBMRegressor(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42,
        verbosity=-1,
    )
    model.fit(train_feat[feature_cols], train_feat["load_MW"])
    return model.predict(test_feat[feature_cols])

print("Running LightGBM backtest...")
lgb_result = run_backtest(df, lightgbm_forecast, step=24*14)  # every 2 weeks — LightGBM is much faster per fold than SARIMA, so more folds is affordable
lgb_summary = summarize(lgb_result, "LightGBM")
print(lgb_result.fold_metrics)
lgb_summary

## 3. Compare against Week 2 baselines

The baseline numbers below are carried over from `02_baselines.ipynb`
(SARIMA ~4.5%, Seasonal Naive ~9.5%, Naive ~20%) rather than recomputed here
— re-running SARIMA's full backtest inside this notebook would just cost
time for a number we already have. If you want a single authoritative
source instead of copy-pasted numbers, consider having `02_baselines.ipynb`
save its `comparison` table to `reports/baseline_results.csv` and loading
it here.


In [ ]:
baseline_scores = pd.DataFrame([
    {"model": "Naive", "mean_mape": 20.0},
    {"model": "Seasonal Naive (168h)", "mean_mape": 9.5},
    {"model": "SARIMA", "mean_mape": 4.5},
])

comparison = pd.concat([
    baseline_scores,
    pd.DataFrame([{"model": "LightGBM", "mean_mape": lgb_summary["mean_mape"]}])
], ignore_index=True).sort_values("mean_mape")

comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(comparison["model"], comparison["mean_mape"])
ax.set_ylabel("Mean MAPE (%)")
ax.set_title("All Models Compared")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 4. Sample fold — predicted vs actual

Same diagnostic as the baselines notebook: a metric alone doesn't show
*how* a model is wrong.


In [ ]:
sample_fold = lgb_result.predictions[lgb_result.predictions["fold"] == 0]

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(sample_fold["timestamp"], sample_fold["y_true"], label="Actual")
ax.plot(sample_fold["timestamp"], sample_fold["y_pred"], label="LightGBM Prediction")
ax.legend()
ax.set_title("LightGBM — Sample Fold")
plt.tight_layout()
plt.show()

## 5. Feature importance (built-in)

LightGBM's own importance score — a quick first look before the more
rigorous SHAP analysis below. This counts how often each feature was used
to split a tree, which is fast but can be a bit crude (it doesn't capture
*how much* each split mattered to the final prediction).


In [ ]:
# Train one model on the full dataset (not a backtest fold) purely for
# interpretability — we want to explain what the model generally learned,
# not a single fold's behavior.
full_feat = build_feature_set(df)
feature_cols = [c for c in full_feat.columns if c not in FEATURE_DROP_COLS]

final_model = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=31, random_state=42, verbosity=-1)
final_model.fit(full_feat[feature_cols], full_feat["load_MW"])

importance = pd.Series(final_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 6))
importance.plot(kind="barh", ax=ax)
ax.invert_yaxis()
ax.set_title("LightGBM Feature Importance (split count)")
plt.tight_layout()
plt.show()

## 6. SHAP — what's actually driving predictions

SHAP values explain *how much* and *in which direction* each feature pushed
an individual prediction, which is more rigorous than a raw importance
count. This is the section worth screenshotting for your README/report —
it's the clearest evidence of *why* the model works, not just that it does.


In [ ]:
# SHAP on a sample of rows for speed (TreeExplainer is exact for LightGBM,
# but computing it for all ~24k rows is unnecessary for a summary plot).
sample = full_feat[feature_cols].sample(min(2000, len(full_feat)), random_state=42)

explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(sample)

shap.summary_plot(shap_values, sample, show=False)
plt.tight_layout()
plt.show()

**What to look for:** temperature-related features and the daily lag
features should dominate, consistent with the EDA findings from
`01_eda.ipynb`. If a feature you expected to matter (e.g. `is_holiday`)
shows up with near-zero SHAP impact, that's consistent with the "holiday
effect was noisy" finding from the EDA notebook — worth cross-referencing
explicitly in your write-up rather than treating them as unrelated results.


## Summary

_(Fill in after running:)_

- Did LightGBM beat SARIMA's ~4.5% MAPE? By how much?
- What did SHAP identify as the top drivers — does it match the EDA's
  temperature/lag findings?
- Does the sample-fold plot show LightGBM handling the kind of sudden
  regime shift (e.g. monsoon-like drops) that seasonal naive struggled with
  in Week 2 — or does it still lag behind fast changes?

## Next steps

`04_deep_learning.ipynb`: LSTM and/or a modern architecture (N-BEATS /
Temporal Fusion Transformer) with quantile/conformal prediction intervals —
the first models in this project that produce uncertainty bounds, not just
point forecasts.
